# Stochastic Q-value iteration

Upload this file to GitHub to view it. To run it, open [Google Colab](https://colab.research.google.com/), select **GitHub** in the open dialog, and paste the notebook link (or upload the file). Run the cells from top to bottom. A CPU is sufficient; the first code cell installs the dependencies. This notebook is self-contained and does not require the `.py` files. You can also use it in Jupyter.

4×4 Gridworld: start at (2, 1), treasure at (4, 4) with reward +1, and fire at (4, 3) with reward −1. Coordinates start at the bottom-left corner. The chosen action succeeds with probability 0.8; each perpendicular direction has probability 0.1. Hitting a wall leaves the agent in the same cell. Rewards belong to the current state R(s). Terminal rewards are received once, with no continuation: Q(terminal, a) = R(terminal). Terminal Q values for every action slot are initialized and fixed at +1 and −1; all other values start at zero.

In [ ]:
%pip install -q matplotlib
%matplotlib inline

## Parameters

Modify `GAMMA`, `ITERATIONS`, and `PLOT_EVERY`, then rerun the following cells.

In [ ]:
SIZE = 4
GAMMA = 0.9  # Discount future rewards: reaching the treasure sooner is better.
ITERATIONS = 10
PLOT_EVERY = 1
START = (2, 1)
TERMINALS = {(4, 4): 1.0, (4, 3): -1.0}
ACTIONS = [(0, 1), (0, -1), (-1, 0), (1, 0)]
SUCCESS_PROB = 0.8
STATES = [(x, y) for y in range(1, SIZE + 1) for x in range(1, SIZE + 1)]



## Model and Bellman update

$Q(s,a)$ is the expected discounted return starting in state $s$, choosing action $a$, and acting optimally afterward. The reward is $R(s)$, attached to the current state.

For nonterminal states:

$$Q_{k+1}(s,a) = R(s) + \gamma \sum_{s'} P(s' \mid s,a) \max_b Q_k(s',b).$$

Every update uses the previous iteration. For each possible successor, first choose the best future action, then average over the transition probabilities. This is planning with a known model, not Q-learning from sampled transitions.

Terminal states have no continuation. We store $Q_k(s,a)=R(s)$ for every terminal action slot as a boundary convention; no action is actually executed there and the terminal payoff is received once. Initially, all nonterminal Q values are zero.

The corresponding state value is $V_k(s)=\max_a Q_k(s,a)$.


In [ ]:
def transition(state, action):
    """Return the next state and current-state reward R(s) for one movement."""
    if state in TERMINALS:
        return state, TERMINALS[state]  # Terminal payoff; no continuation in Bellman.
    x, y = state
    dx, dy = action
    next_state = (min(SIZE, max(1, x + dx)), min(SIZE, max(1, y + dy)))
    reward = TERMINALS.get(state, 0.0)
    return next_state, reward


def outcomes(state, action):
    """Yield (probability, next state, reward) for the three possible moves."""
    dx, dy = action
    slip_prob = (1.0 - SUCCESS_PROB) / 2.0
    for probability, movement in [
        (SUCCESS_PROB, action),
        (slip_prob, (-dy, dx)),  # Rotate the chosen direction 90 degrees.
        (slip_prob, (dy, -dx)),  # Rotate it -90 degrees.
    ]:
        next_state, reward = transition(state, movement)
        yield probability, next_state, reward


def initial_q_values():
    """Use terminal payoffs as boundary values for every action slot."""
    return {(state, action): TERMINALS.get(state, 0.0)
            for state in STATES for action in ACTIONS}


def bellman_update(q_values):
    """Update all state-action pairs synchronously using the known model."""
    new_q_values = {}
    for state in STATES:
        for action in ACTIONS:
            if state in TERMINALS:
                # No actions are executed here: these slots store the final payoff.
                new_q_values[state, action] = TERMINALS[state]
                continue
            # Q_new(s,a) = R(s) + gamma * sum P(s'|s,a) * max_b Q_old(s',b).
            # Choose the best future action separately for each successor.
            # Average over exact probabilities; no sampled experience is used.
            new_q_values[state, action] = sum(
                probability * (reward + GAMMA * max(
                    q_values[next_state, next_action] for next_action in ACTIONS
                ))
                for probability, next_state, reward in outcomes(state, action)
            )
    return new_q_values


## Visualization

Each cell shows four Q values, positioned up, down, left, and right. Best actions (including ties) are highlighted in green. Terminal cells show their fixed payoff.


In [ ]:
def plot_q_values(ax, q_values, iteration):
    """Show each action value in its movement direction inside the cell."""
    from matplotlib.patches import Rectangle

    ax.clear()
    for x, y in STATES:
        state = (x, y)
        color = "white"
        if state in TERMINALS:
            color = "#d8f3dc" if TERMINALS[state] > 0 else "#ffdad6"
        elif state == START:
            color = "#dceeff"
        ax.add_patch(Rectangle((x - 0.5, y - 0.5), 1, 1,
                               facecolor=color, edgecolor="black"))
        if state in TERMINALS:
            ax.text(x, y, f"Terminal\nQ = R = {TERMINALS[state]:+.0f}",
                    ha="center", va="center", fontsize=10)
            continue
        ax.plot([x - 0.5, x + 0.5], [y - 0.5, y + 0.5], color="0.8", lw=0.7)
        ax.plot([x - 0.5, x + 0.5], [y + 0.5, y - 0.5], color="0.8", lw=0.7)
        best = max(q_values[state, action] for action in ACTIONS)
        for action in ACTIONS:
            dx, dy = action
            value = q_values[state, action]
            is_best = abs(value - best) < 1e-10
            ax.text(x + 0.32 * dx, y + 0.32 * dy, f"{value:.3f}",
                    ha="center", va="center", fontsize=9,
                    color="#126b36" if is_best else "black",
                    fontweight="bold" if is_best else "normal")
        if state == START:
            ax.text(x, y, "S", ha="center", va="center", fontsize=9)
    ax.set(xlim=(0.5, SIZE + 0.5), ylim=(0.5, SIZE + 0.5),
           xticks=range(1, SIZE + 1), yticks=range(1, SIZE + 1),
           xlabel="x", ylabel="y",
           title=f"Stochastic Q-value iteration — sweep {iteration}\n"
                 f"gamma = {GAMMA}, action success = {SUCCESS_PROB:.0%}\n"
                 "Q values: up / down / left / right; best actions in green")
    ax.set_aspect("equal")


## Run Q-value iteration

Display the initial values and selected iterations directly in the notebook.


In [ ]:
import matplotlib.pyplot as plt

assert ITERATIONS >= 1 and PLOT_EVERY >= 1
q_values = initial_q_values()
q_history = [q_values.copy()]
deltas = []

def show_q_values(q_values, iteration):
    fig, ax = plt.subplots(figsize=(9, 9))
    plot_q_values(ax, q_values, iteration)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

show_q_values(q_values, 0)
for iteration in range(1, ITERATIONS + 1):
    new_q_values = bellman_update(q_values)
    delta = max(abs(new_q_values[key] - q_values[key]) for key in q_values)
    q_values = new_q_values
    q_history.append(q_values.copy())
    deltas.append(delta)
    if iteration % PLOT_EVERY == 0 or iteration == ITERATIONS:
        print(f"Sweep {iteration:3d}: max Q-value change = {delta:.6f}")
        show_q_values(q_values, iteration)


## Convergence

If the final change is still large, increase `ITERATIONS`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(1, ITERATIONS + 1), deltas, marker="o")
ax.set(xlabel="Iteration", ylabel="Maximum Q-value change", title="Convergence")
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)